# Compute Article Embeddings (run this notebook on Kaggle)

This notebook is **not** part of the local one-command pipeline -- it needs a
GPU, so it's meant to be run standalone on Kaggle, with the results
downloaded back into the local repo. See `SPEC.md` Q3 section 1 for why
(GPU-backed compute for the embedding model itself; the local repo only ever
does inference/retrieval math on the resulting vectors, no ML library needed
there).

**Steps:**
1. Locally, copy `data/processed/ebnerd/articles.parquet` and
   `data/processed/mind/articles.parquet` somewhere convenient and rename
   them `ebnerd_articles.parquet` / `mind_articles.parquet` (so their names
   are unambiguous once uploaded together).
2. On [kaggle.com](https://www.kaggle.com), create a new Notebook, then in
   the right sidebar click **+ Add Data -> Upload -> New Dataset** and upload
   both renamed files together as one private dataset (e.g. named
   `cs4406-articles`).
3. Upload this notebook (File -> Import Notebook) or paste its cells into a
   new one.
4. In the right sidebar: **Settings -> Accelerator -> GPU T4 x2** (or P100),
   and **Settings -> Internet -> On** (needed for `pip install` and the
   HuggingFace model download; off by default on Kaggle).
5. **Save Version -> Save & Run All (Commit)**. Once it finishes, open that
   version and go to its **Output** tab to download
   `ebnerd_article_embeddings.parquet` and `mind_article_embeddings.parquet`.
6. Move them into your local repo as:
   - `data/processed/ebnerd/article_embeddings.parquet`
   - `data/processed/mind/article_embeddings.parquet`
7. Continue locally with `embedding_retrieval.ipynb` / `python
   embedding_retrieval.py` -- it only reads these two files, no GPU or ML
   dependency required on your machine.

Model: `paraphrase-xlm-r-multilingual-v1` (XLM-RoBERTa-based, 768-dim,
handles Danish and English in one model -- matches the assignment's
"BERT/XLM-RoBERTa" suggestion for Q3).

In [ ]:
!pip install -q sentence-transformers

## Locate the uploaded `articles.parquet` files

Auto-discovers both files under `/kaggle/input/` by filename pattern. If this
fails (nothing found, or more than one match), check the exact path shown in
the Kaggle "Data" sidebar for your attached dataset and set
`EBNERD_ARTICLES_PATH` / `MIND_ARTICLES_PATH` manually below.

In [ ]:
from glob import glob

import pandas as pd


def find_one(pattern: str) -> str:
    matches = glob(pattern, recursive=True)
    if len(matches) != 1:
        raise FileNotFoundError(
            f"expected exactly one match for {pattern!r}, found {matches}. "
            "Check the Data sidebar for the real path and set the path manually."
        )
    return matches[0]


EBNERD_ARTICLES_PATH = find_one("/kaggle/input/**/*ebnerd*articles*.parquet")
MIND_ARTICLES_PATH = find_one("/kaggle/input/**/*mind*articles*.parquet")
print("ebnerd:", EBNERD_ARTICLES_PATH)
print("mind:", MIND_ARTICLES_PATH)

ebnerd_articles = pd.read_parquet(EBNERD_ARTICLES_PATH)
mind_articles = pd.read_parquet(MIND_ARTICLES_PATH)
print("ebnerd:", ebnerd_articles.shape, "| mind:", mind_articles.shape)

In [ ]:
assert {"article_id", "title", "abstract"}.issubset(ebnerd_articles.columns)
assert {"article_id", "title", "abstract"}.issubset(mind_articles.columns)
assert ebnerd_articles["article_id"].str.startswith("ebnerd_").all()
assert mind_articles["article_id"].str.startswith("mind_").all()
print("ok: uploaded files match the expected unified schema (SPEC.md Q1 section 2)")

## Load the multilingual model onto GPU

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("WARNING: no GPU detected -- Settings > Accelerator > GPU for reasonable speed.")

MODEL_NAME = "paraphrase-xlm-r-multilingual-v1"
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
EMBEDDING_DIM = model.get_sentence_embedding_dimension()
print("device:", DEVICE, "| embedding dim:", EMBEDDING_DIM)

In [ ]:
assert EMBEDDING_DIM > 0
print("ok: multilingual model loaded")

## Encode `title + abstract` for both datasets

Same corpus-text convention as Q2/Q3 (`title + " " + abstract`, `abstract`
`.fillna("")`'d -- see `SPEC.md` Q2 section 3 for why the fillna matters).

In [ ]:
def build_corpus_text(articles: pd.DataFrame) -> list[str]:
    return (articles["title"].fillna("") + " " + articles["abstract"].fillna("")).tolist()


ebnerd_texts = build_corpus_text(ebnerd_articles)
mind_texts = build_corpus_text(mind_articles)

ebnerd_embeddings = model.encode(ebnerd_texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True)
mind_embeddings = model.encode(mind_texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True)

print("ebnerd embeddings:", ebnerd_embeddings.shape, "| mind embeddings:", mind_embeddings.shape)

In [ ]:
import numpy as np

assert ebnerd_embeddings.shape == (len(ebnerd_articles), EMBEDDING_DIM)
assert mind_embeddings.shape == (len(mind_articles), EMBEDDING_DIM)
assert not np.isnan(ebnerd_embeddings).any()
assert not np.isnan(mind_embeddings).any()
print("ok: embeddings computed for both datasets, correct shape, no NaNs")

## Save to `/kaggle/working/`

Output schema matches `SPEC.md` Q3 section 6 (`article_id, dataset,
embedding`). Anything written to `/kaggle/working/` becomes downloadable
from this version's **Output** tab after Save & Run All completes.

In [ ]:
def save_embeddings(articles: pd.DataFrame, embeddings: np.ndarray, dataset: str, out_path: str) -> str:
    df = pd.DataFrame({
        "article_id": articles["article_id"].to_numpy(),
        "dataset": dataset,
        "embedding": [row.astype("float32").tolist() for row in embeddings],
    })
    df.to_parquet(out_path, index=False)
    return out_path


ebnerd_out = save_embeddings(
    ebnerd_articles, ebnerd_embeddings, "ebnerd", "/kaggle/working/ebnerd_article_embeddings.parquet"
)
mind_out = save_embeddings(
    mind_articles, mind_embeddings, "mind", "/kaggle/working/mind_article_embeddings.parquet"
)
print("wrote:", ebnerd_out)
print("wrote:", mind_out)

## Next steps

After this notebook finishes running (Save Version -> Save & Run All), open
that version's **Output** tab and download:
- `ebnerd_article_embeddings.parquet` -> place at
  `data/processed/ebnerd/article_embeddings.parquet` in your local repo
- `mind_article_embeddings.parquet` -> place at
  `data/processed/mind/article_embeddings.parquet` in your local repo

Then run `embedding_retrieval.ipynb` / `python embedding_retrieval.py`
locally.